# WAXAL ASR — Fine-Tune Gemma 3n for African Language Speech Recognition

This notebook provides an interactive walkthrough of the full training pipeline for the
[Google Research WAXAL African Language ASR Challenge](https://zindi.africa/).

**What this notebook does:**
1. Installs and verifies dependencies
2. Loads configuration from YAML files
3. Authenticates with HuggingFace Hub
4. Loads and inspects the WaxalNLP dataset
5. Loads the Gemma 3n model with LoRA adapters
6. Fine-tunes the model using TRL's SFTTrainer
7. Evaluates with WER and CER metrics
8. Generates a Zindi submission CSV

**Hardware:** A100 (40 GB) recommended. T4 (16 GB) works with gradient checkpointing.

**Model:** `google/gemma-3n-E2B-it` (~2B effective parameters) with native audio input.

---

## Background: WaxalNLP Dataset

The [WaxalNLP dataset](https://huggingface.co/datasets/google/WaxalNLP) covers
**27 Sub-Saharan African languages** with ~1,846 hours of transcribed ASR data,
licensed under CC-BY-4.0. This competition focuses on three languages:

| Language | Code | Train Examples |
|----------|------|---------------|
| Luganda  | `lug` | 2,602 |
| Lingala  | `lin` | 9,937 |
| Shona    | `sna` | 10,489 |

Each example contains:
- `audio`: raw waveform (decoded by HuggingFace datasets)
- `transcription`: ground-truth text

## 1. Install Dependencies

We install all packages listed in `requirements.txt`. The `%%capture` magic
suppresses verbose pip output to keep the notebook clean.

In [ ]:
%%capture
# Install all project dependencies from the requirements file.
# This ensures version consistency between notebook and CLI usage.
!pip install -q -r ../requirements.txt

print("Dependencies installed.")

## 2. Environment Setup and Configuration

### Why configuration files?

Hardcoded hyperparameters make experiments difficult to track and reproduce.
By loading all settings from a YAML file, we can:
- Version-control exact experiment configurations
- Quickly switch between languages or hyperparameter sweeps
- Share reproducible setups with collaborators

### Expected output
- Software versions printed for reproducibility
- GPU information displayed
- Configuration loaded and shown

In [ ]:
import sys
from pathlib import Path

# Add the project root to the Python path so we can import from src/
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import load_config
from src.utils import setup_logging, set_seed, log_environment, log_gpu_info, log_memory_usage

# Initialise logging — all modules will use Python's logging framework
setup_logging()

# Load configuration — change the path below to switch languages or experiments
# Options: configs/default.yaml, configs/sna.yaml, configs/lug.yaml, configs/lin.yaml
CONFIG_OVERRIDE = None  # Set to e.g. "../configs/sna.yaml" for Shona-only
config = load_config(override_path=CONFIG_OVERRIDE)

# Set random seeds for reproducibility across Python, NumPy, and PyTorch
set_seed(config["seed"])

# Print environment details for reproducibility records
log_environment()
log_gpu_info()

# Display key config values
print(f"\nModel     : {config['model']['model_id']}")
print(f"Languages : {config['dataset']['languages']}")
print(f"Max steps : {config['training']['max_steps']}")
print(f"Output    : {config['training']['output_dir']}")

## 3. Authenticate with HuggingFace

### Why authentication?

The Gemma model family requires accepting a license agreement on the
[model card](https://huggingface.co/google/gemma-3n-E2B-it) before download.
You need a [HuggingFace access token](https://huggingface.co/settings/tokens)
with **read** permissions.

### Expected output
- Interactive login prompt (in Colab) or confirmation of existing token

In [ ]:
from huggingface_hub import login

# Interactive login — you will be prompted for your HF token
login()

## 4. Load the WaxalNLP Dataset

### Why streaming?

The full WaxalNLP dataset is large. Streaming mode avoids downloading
everything upfront — examples are fetched and decoded on-the-fly as the
trainer iterates over them.

### Chat formatting

Gemma 3n is an instruction-tuned model that expects input in a chat format
with system, user, and assistant turns. Each audio example is wrapped into
this format, where:
- **System turn**: instructs the model to transcribe accurately
- **User turn**: contains the audio waveform and a transcription request
- **Assistant turn**: contains the ground-truth transcription (for training)

### Expected output
- Dataset loaded for all configured languages
- One sample previewed with transcription text and audio shape

In [ ]:
import numpy as np

from src.dataset import load_datasets_from_config, log_dataset_statistics

# Load train, validation, and test splits for all configured languages
splits = load_datasets_from_config(config)
train_ds = splits["train"]
val_ds = splits["validation"]
test_ds = splits["test"]

# Peek at dataset statistics (audio duration, transcription length)
log_dataset_statistics(train_ds, "train", num_peek=50)

# Preview one example
sample = next(iter(train_ds))
print(f"\nTranscription: {sample['transcription']}")
print(f"Audio shape  : {np.array(sample['audio']['array']).shape}")
print(f"Sample rate  : {sample['audio']['sampling_rate']} Hz")

## 5. Load Model and Processor

### Model architecture

Gemma 3n is a multimodal model that natively accepts audio input alongside
text. It uses a shared embedding space for audio and text tokens, which means
we do not need a separate audio encoder — the model handles end-to-end
speech-to-text.

We load the model in `bfloat16` precision to reduce VRAM usage while
maintaining training stability (bfloat16 has the same exponent range as
float32, unlike float16).

### Expected output
- Model and processor loaded
- Device and dtype confirmed
- GPU memory usage reported

In [ ]:
from src.model import load_model_and_processor

# Load the pretrained model and its tokenizer/processor
model, processor = load_model_and_processor(config)
log_memory_usage()

## 6. Fine-Tune with LoRA (PEFT + TRL SFT)

### Why LoRA?

Low-Rank Adaptation (LoRA) inserts small trainable rank-decomposition matrices
into selected attention projections while keeping the rest of the model frozen.
This reduces the number of trainable parameters from billions to millions,
making fine-tuning feasible on a single GPU.

**Key hyperparameters:**

| Parameter | Value | Effect |
|-----------|-------|--------|
| `r` | 8 | Rank of the decomposition matrices. Higher = more capacity |
| `alpha` | 16 | Scaling factor. Effective scale = alpha / r |
| `target_modules` | `v_proj, o_proj` | Which attention projections to adapt |
| `max_steps` | 500 | Training iterations (increase to ~3000 for production) |

### Data collation

The collator tokenises each batch on-the-fly and masks padding and special
tokens in the labels (setting them to -100) so the cross-entropy loss is
only computed on real transcription tokens.

### Expected output
- Training progress bar with loss values
- Periodic evaluation losses

In [ ]:
from src.trainer import build_trainer

training_cfg = config["training"]
seed = config["seed"]

# Shuffle training data with a buffer and repeat indefinitely for streaming
shuffled_train = train_ds.shuffle(
    buffer_size=training_cfg.get("shuffle_buffer_size", 1000),
    seed=seed,
).repeat(None)

# Take a fixed validation slice for consistent evaluation across runs
num_val = training_cfg.get("num_validation_examples", 200)
val_ds_fixed = val_ds.take(num_val)

# Build the trainer with LoRA config and training arguments
trainer = build_trainer(
    model=model,
    processor=processor,
    train_dataset=shuffled_train,
    eval_dataset=val_ds_fixed,
    config=config,
)

# Start fine-tuning
trainer.train()
print("Fine-tuning complete.")
log_memory_usage()

## 7. Evaluate: Word Error Rate (WER) and Character Error Rate (CER)

### Metrics explained

- **WER** (Word Error Rate): measures the edit distance at the word level.
  WER = (substitutions + deletions + insertions) / total reference words.
  This is the primary ASR metric.

- **CER** (Character Error Rate): same formula but at the character level.
  CER is especially important for morphologically rich African languages
  where word boundaries can be ambiguous.

Lower values are better for both metrics.

### Expected output
- WER and CER percentages on the test set
- Metrics saved to JSON file
- Sample predictions saved for manual inspection

In [ ]:
from tqdm import tqdm

from src.inference import transcribe_batch
from src.metrics import compute_all_metrics, save_metrics, save_prediction_examples
from src.utils import get_device

eval_cfg = config["evaluation"]
device = get_device()
model.eval()

# Collect references and predictions from the test split
references = []
predictions = []
num_samples = eval_cfg["num_samples"]
batch_size = eval_cfg["batch_size"]
num_batches = (num_samples + batch_size - 1) // batch_size

for batch in tqdm(test_ds.take(num_samples).batch(batch_size=batch_size), total=num_batches, desc="Evaluating"):
    preds = transcribe_batch(batch, model, processor, device, eval_cfg["max_new_tokens"])
    references.extend(batch["transcription"])
    predictions.extend(preds)

# Compute metrics
metrics = compute_all_metrics(references, predictions)
languages = config["dataset"]["languages"]
lang_str = "-".join(languages)

print(f"\nResults on {lang_str} test set:")
print(f"  WER : {metrics['wer']:.2%}")
print(f"  CER : {metrics['cer']:.2%}")

# Save metrics and prediction examples for later analysis
output_dir = Path(config["training"]["output_dir"]) / "eval"
output_dir.mkdir(parents=True, exist_ok=True)
save_metrics(metrics, output_dir / "metrics.json")
save_prediction_examples(references, predictions, output_dir / "predictions.json")

## 8. Generate Submission

### Submission format

The Zindi platform expects a CSV with two columns:
- `ID`: the test example identifier (e.g. `lug_96114`)
- `Target`: the predicted transcription

The ID prefix encodes the language, which we use to load the correct
test split from HuggingFace.

### Expected output
- `submission.csv` generated and validated against the sample submission

In [ ]:
# For submission generation, run the standalone script:
# !python ../submit.py --config ../configs/default.yaml
#
# Or run it directly from the notebook:
import subprocess
result = subprocess.run(
    [sys.executable, str(project_root / "submit.py")],
    cwd=str(project_root),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 9. Next Steps

To improve results beyond the baseline:

1. **Increase training steps** — set `max_steps: 3000` in the config
2. **Expand LoRA targets** — add `q_proj`, `k_proj` to `target_modules`
3. **Increase LoRA rank** — try `r: 16` or `r: 32`
4. **Multilingual training** — use `configs/multilingual.yaml` to train on all three languages
5. **Learning rate scheduling** — experiment with cosine annealing
6. **Data augmentation** — add noise or speed perturbation to audio
7. **Larger model** — try `google/gemma-4n-E4B-it` on an A100